In [1]:
import os
import json
import pickle
import pathlib
import numpy as np
import scipy.interpolate as interp
from tqdm import tqdm # Used for the loading progress bar

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# Set your directory
dir = r"/Volumes/ESSD/BatteryLife" if os.name == 'posix' else r"D:\BatteryLife"

# Function to extract labels
def get_dict(labels_dir):
    dict_labels = {}
    content = os.listdir(labels_dir)
  
    for f in content:
        if f.startswith("._") or f.startswith(".DS_Store"):
            continue
        with open(os.path.join(labels_dir, f), 'r') as file:
            data = json.load(file)
            if 'Tongji' in f:
               temp_dict = {}
               for k, v in data.items():
                    k_new = k.replace("#", "-")
                    temp_dict.update({k_new:v})
               data = temp_dict   
            dict_labels.update(data)
            
    k = list(dict_labels.keys())
    v = list(dict_labels.values())
    return k, v

labels_path = os.path.join(dir, "Life labels")
cycles_file, soh = get_dict(labels_path)
print(f"Found {len(soh)} labels.")

Found 1208 labels.


In [3]:

def get_length(dir):
    content = os.listdir(dir)
    length = 0
    for f in content:
        if f.startswith("._") or f.startswith(".DS_Store"):
            continue
        with open(os.path.join(dir, f), 'r') as file:
            data = json.load(file)
            length += len(data)
    return length


        

In [4]:
content = os.listdir(dir)
print(f"###### \n ---- Content ---- \n{content} \n######")
folders = [f for f in content if os.path.isdir(os.path.join(dir, f)) and (f != "Life labels" and f != "READMEs") and not f.startswith("._") and not f.startswith(".DS_Store")]
pkl_files = [[] for _ in folders]
for i, folder in enumerate(folders) :
    folder_path = os.path.join(dir, folder)
    for file in os.listdir(folder_path):
        if file.endswith('.pkl'):
            pkl_files[i].append(file)
print(len(pkl_files))

###### 
 ---- Content ---- 
['CALB', 'CALCE', 'HNEI', 'HUST', 'ISU_ILCC', 'Life labels', 'MATR', 'MICH', 'MICH_EXP', 'NA-ion', 'README.md', 'READMEs', 'RWTH', 'SDU', 'SNL', 'Stanford', 'Stanford_2', 'Tongji', 'UL_PUR', 'XJTU', 'ZN-coin', '.DS_Store', '._.DS_Store', '._README.md'] 
######
18


In [5]:
labels = os.path.join(dir, "Life labels")
def get_dict(labels):
    dict_labels = {}
    content = os.listdir(labels)
    length = 0
  
    for f in content:
        if f.startswith("._") or f.startswith(".DS_Store"):
          continue
        with open(os.path.join(labels, f), 'r') as file:
            data = json.load(file)
            if 'Tongji' in f:
               temp_dict = {}
               for k, v in data.items():
                    k_new = k.replace("#", "-")
                    temp_dict.update({k_new:v})
               data = temp_dict   
            dict_labels.update(data)
    k= list(dict_labels.keys())
    v= list(dict_labels.values())
    return k,v


In [25]:
import numpy as np
import scipy.interpolate as interp

def get_cycle_data(data, cycle_format):
    
    arr_total = np.zeros((cycle_format, 3, 300))
    
    for i in range(cycle_format):

        raw_time = np.array(data['cycle_data'][i]['time_in_s'])
        scaled_time = raw_time - raw_time[0]
        raw_current = np.array(data['cycle_data'][i]['current_in_A'])
        raw_voltage = np.array(data['cycle_data'][i]['voltage_in_V'])
        dt = np.diff(scaled_time)
        avg_current = 0.5 * (raw_current[1:] + raw_current[:-1])
        
        cumulative_charge = np.concatenate((
            [0],
            np.cumsum(avg_current * dt)
        ))
        
        capacity_Ah = cumulative_charge / 3600.0

        ## Some of the cycles I saw had starting in charging and others where starting in dishcharging, which mean the capacity values would be nonsesical
        ## thus checks are done below

        # If index of starting to charge is higher (after discharge), the chances are its discharging from 100% SoC first, so deduct peak capacity intergral from running intergral
        if np.searchsorted(capacity_Ah, 0.01) > np.searchsorted(capacity_Ah, -0.01):
            capacity_Ah = capacity_Ah.max() + capacity_Ah
    

        t_new = np.linspace(0, scaled_time[-1], num=300)
        f_current = interp.interp1d(scaled_time, raw_current, kind='linear')
        f_voltage = interp.interp1d(scaled_time, raw_voltage, kind='linear')
        f_capacity = interp.interp1d(scaled_time, capacity_Ah, kind='linear')
        current_interpolated = f_current(t_new) / data['nominal_capacity_in_Ah']
        voltage_interpolated = f_voltage(t_new) / data['max_voltage_limit_in_V']
        capacity_interpolated = f_capacity(t_new) / data['nominal_capacity_in_Ah']
        
        arr_total[i] = np.stack(
            (current_interpolated,
             voltage_interpolated,
             capacity_interpolated),
            axis=0
        )
    
    return arr_total


In [26]:
test = np.array([1,2,3,4,5,6,7,8,9])
test[1:] + test[:-1]

array([ 3,  5,  7,  9, 11, 13, 15, 17])

In [30]:
# Set up a local folder to save the fast-loading tensors
local_save_dir = "./processed_battery_data" 
os.makedirs(local_save_dir, exist_ok=True)

x_path = os.path.join(local_save_dir, "X_features.pt")
y_path = os.path.join(local_save_dir, "y_labels.pt")

# Check if we already processed and saved the data locally
if os.path.exists(x_path) and os.path.exists(y_path):
    print("Loading pre-processed tensors from local storage...")
    X_tensor = torch.load(x_path)
    y_tensor = torch.load(y_path)
    print("Loaded successfully!")

else:
    print("Tensors not found locally. Extracting from external drive (this will take a few minutes)...")
    
    # Fast path mapping: Find all paths once so we don't have to search later
    file_paths = {}
    for loc in pathlib.Path(dir).rglob('*.pkl'):
        file_paths[loc.name] = loc

    all_features = []
    all_labels = []

    # Loop through all files and extract the features
    for idx in tqdm(range(len(soh))):
        file_name = cycles_file[idx]
        if file_name not in file_paths:
            print(f"Skipping missing file: {file_name}")
            continue
            
        with open(file_paths[file_name], 'rb') as file:
            data = pickle.load(file)

        features = get_cycle_data(data, cycle_format=1)
        all_features.append(features)
        all_labels.append(soh[idx])

    # Convert lists to PyTorch tensors
    X_tensor = torch.tensor(np.array(all_features), dtype=torch.float32)
    y_tensor = torch.tensor(np.array(all_labels), dtype=torch.float32)
    
    # Save them locally for next time!
    torch.save(X_tensor, x_path)
    torch.save(y_tensor, y_path)
    print(f"Saved tensors to {local_save_dir}!")

print(f"Total Features Shape: {X_tensor.shape}")

# --- NORMALIZATION ---
# Find the maximum cycle life in your dataset
y_max = y_tensor.max()
print(f"Max cycle life in dataset: {y_max.item()} cycles")

# Scale all targets to be between 0 and 1
y_tensor_normalized = y_tensor / y_max

# Create the dataset using the NORMALIZED labels
dataset = TensorDataset(X_tensor, y_tensor_normalized)
print(f"Size of dataset: {len(dataset)}")

# Safely split data (1000 for training, the rest for validation)
train_size = 1000
val_size = len(dataset) - train_size
train_data, val_data = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=0)
validation_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=0)

Tensors not found locally. Extracting from external drive (this will take a few minutes)...


100%|██████████| 1208/1208 [06:47<00:00,  2.97it/s]

Saved tensors to ./processed_battery_data!
Total Features Shape: torch.Size([1208, 1, 3, 300])
Max cycle life in dataset: 4999.0 cycles
Size of dataset: 1208


In [31]:
train_features, train_labels = next(iter(train_loader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels}")
print(f"Squeezed shape: {train_features.squeeze(1).shape}")

Feature batch shape: torch.Size([64, 1, 3, 300])
Labels batch shape: tensor([0.0278, 0.0588, 0.1688, 0.1150, 0.0370, 0.2478, 0.1046, 0.0996, 0.0500,
        0.7874, 0.0732, 0.0226, 0.3453, 0.1710, 0.0988, 0.0394, 0.4719, 0.0790,
        0.1870, 0.0746, 0.0952, 0.0286, 0.4707, 0.0624, 0.0524, 0.6647, 0.1286,
        0.0666, 0.1308, 0.0350, 0.0640, 0.2841, 0.1682, 0.1680, 0.0240, 0.1178,
        0.1074, 0.0658, 0.0318, 0.0846, 0.1122, 0.1772, 0.1242, 0.2669, 0.0426,
        0.0438, 0.2539, 0.0826, 0.0208, 0.1272, 0.1428, 0.1556, 0.1682, 0.0226,
        0.7538, 0.1158, 0.1548, 0.2024, 0.1444, 0.1364, 0.3085, 0.1994, 0.2709,
        0.1094])
Squeezed shape: torch.Size([64, 3, 300])


In [33]:
import torch
import torch.nn as nn

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device}, gpu -> {torch.cuda.get_device_name(0) if torch.cuda.is_available() is True else "N/A" }")


y_max = y_max.to(device)

class LSTMnetwork(nn.Module):
    def __init__(self, hidden_size=80, num_layers=4):
        super(LSTMnetwork, self).__init__()

        # Upgraded to LSTM, increased hidden size, and added a layer
        self.lstm = nn.LSTM(
            input_size=3, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=0.3 
        )

        self.out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = x.squeeze(1).permute(0, 2, 1) 
        _, (hidden_layers, _) = self.lstm(x)
        cycle_to_80_pred = self.out(hidden_layers[-1]) 

        return cycle_to_80_pred.flatten()

model = LSTMnetwork(hidden_size=95, num_layers=5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# ==========================
# 4. TRAINING & VALIDATION LOOP
# ==========================
epochs = 3000

# Lists to keep track of our errors so we can plot them later
train_mae_history = []
val_mae_history = []

for epoch in range(epochs):
    
    # --- TRAINING PHASE ---
    model.train() # Set the model to training mode (enables gradients and dropout)
    train_loss = 0
    train_mae = 0

    for batch_x, batch_y in train_loader:
        # Move data to the Mac GPU
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        
        preds = model(batch_x)   
        
        # Calculate MSE loss on normalized data for the optimizer
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

        # Calculate MAE in real cycle numbers for human readability
        with torch.no_grad():
            preds_real = preds * y_max
            batch_y_real = batch_y * y_max
            mae = torch.abs(preds_real - batch_y_real).mean()
            train_mae += mae.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_train_mae = train_mae / len(train_loader)
    train_mae_history.append(avg_train_mae)

    # --- VALIDATION PHASE ---
    model.eval() # Set model to evaluation mode (turns off dropout, locks weights)
    val_loss = 0
    val_mae = 0

    # torch.no_grad() tells PyTorch not to calculate gradients (saves memory/time)
    with torch.no_grad():
        for batch_x, batch_y in validation_loader:
            # Move data to the Mac GPU
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            preds = model(batch_x)
            
            # Calculate validation loss
            loss = criterion(preds, batch_y)
            val_loss += loss.item()
            
            # Calculate validation MAE in real cycle numbers
            preds_real = preds * y_max
            batch_y_real = batch_y * y_max
            mae = torch.abs(preds_real - batch_y_real).mean()
            val_mae += mae.item()

    avg_val_loss = val_loss / len(validation_loader)
    avg_val_mae = val_mae / len(validation_loader)
    val_mae_history.append(avg_val_mae)

    # --- PRINT RESULTS ---
    print(f"Epoch {epoch:02d} | "
          f"Train Error: {avg_train_mae:.1f} cycles | "
          f"Val Error: {avg_val_mae:.1f} cycles")

Using cuda, gpu -> NVIDIA GeForce RTX 5080
Epoch 00 | Train Error: 501.9 cycles | Val Error: 427.2 cycles
Epoch 01 | Train Error: 456.1 cycles | Val Error: 402.8 cycles
Epoch 02 | Train Error: 428.9 cycles | Val Error: 428.9 cycles
Epoch 03 | Train Error: 409.8 cycles | Val Error: 403.4 cycles
Epoch 04 | Train Error: 379.2 cycles | Val Error: 376.9 cycles
Epoch 05 | Train Error: 369.5 cycles | Val Error: 340.1 cycles
Epoch 06 | Train Error: 366.4 cycles | Val Error: 433.4 cycles
Epoch 07 | Train Error: 365.8 cycles | Val Error: 401.2 cycles
Epoch 08 | Train Error: 360.2 cycles | Val Error: 382.3 cycles
Epoch 09 | Train Error: 356.4 cycles | Val Error: 356.6 cycles
Epoch 10 | Train Error: 350.5 cycles | Val Error: 385.5 cycles
Epoch 11 | Train Error: 346.9 cycles | Val Error: 297.4 cycles
Epoch 12 | Train Error: 319.6 cycles | Val Error: 299.0 cycles
Epoch 13 | Train Error: 338.7 cycles | Val Error: 355.9 cycles
Epoch 14 | Train Error: 325.7 cycles | Val Error: 298.9 cycles
Epoch 15 | T